### Prepapra a base para o GLM para a analise da importancia das palavras chaves

Vamos olhar o quanto as palavras chaves impactam no processo de seleção

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import openai
import numpy as np
import time
import os
from dotenv import load_dotenv, find_dotenv
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
### style de execução
tqdm.pandas()

In [3]:
# 1. Configurar para exibir todas as linhas
pd.set_option('display.max_rows', None)

# 2. Configurar para exibir todas as colunas
pd.set_option('display.max_columns', None)

## Leitura e processamento dos resultados processados

### Openai

In [4]:
df_slr1_titulo_resumo_gpt = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_openai_gpt_4o_slr1_v2.csv", encoding='utf-8')

df_slr1_titulo_keywords_gpt = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_openai_gpt_4o_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_keywords_gpt = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_openai_gpt_4o_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_gpt = pd.read_csv("/data/codigos/dados/resultados/resumo_openai_gpt_4o_slr1_v2.csv", encoding='utf-8')


In [5]:
# traz todos os resultados ja analisados na fase 1
df_slr1_gpt = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_openai_gpt_4o_slr1.csv", encoding='utf-8')
df_slr1_gpt.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,gpt-4o_IC1_0,gpt-4o_IC1_1,gpt-4o_IC1_2,gpt-4o_IC1_3,gpt-4o_IC1_4,gpt-4o_IC2_0,gpt-4o_IC2_1,gpt-4o_IC2_2,gpt-4o_IC2_3,gpt-4o_IC2_4
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não estruturado,7,7,7,7,7,5,5,5,5,6
1,slr1_2,Sentiment analysis on arabic tweets: Challeng...,Sentiment analysis and opinion mining are desi...,"Twitter, Sentiment analysis, Opinion mining, N...",sucesso,sucesso,não estruturado,3,3,3,3,4,3,3,3,3,1


In [6]:
# só mantem a galera com resultados selecionados no sample
df_slr1_gpt["flag_selecao"] = df_slr1_gpt["ID"].isin(df_slr1_titulo_resumo_gpt["ID"].to_list())
df_slr1_gpt = df_slr1_gpt[df_slr1_gpt["flag_selecao"] == True].copy()
df_slr1_gpt = df_slr1_gpt.drop(columns=["flag_selecao"])
df_slr1_gpt.shape

(47, 17)

### Google

In [7]:
df_slr1_resumo_keywords_google = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_gemini_15_flash_slr1_v2.csv", encoding='utf-8')

df_slr1_titulo_resumo_google = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_gemini_15_flash_slr1_v2.csv", encoding='utf-8')

df_slr1_titulo_keywords_google = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_gemini_15_flash_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_google = pd.read_csv("/data/codigos/dados/resultados/resumo_gemini_15_flash_slr1_v2.csv", encoding='utf-8')


In [8]:
# traz todos os resultados ja analisados na fase 1
df_slr1_google = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_gemini_15_flash_slr1.csv", encoding='utf-8')
df_slr1_google.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,gemini-1.5-flash_IC1_0,gemini-1.5-flash_IC1_1,gemini-1.5-flash_IC1_2,gemini-1.5-flash_IC1_3,gemini-1.5-flash_IC1_4,gemini-1.5-flash_IC2_0,gemini-1.5-flash_IC2_1,gemini-1.5-flash_IC2_2,gemini-1.5-flash_IC2_3,gemini-1.5-flash_IC2_4
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não estruturado,7,7,7,7,7,5,5,5,5,5
1,slr1_2,Sentiment analysis on arabic tweets: Challeng...,Sentiment analysis and opinion mining are desi...,"Twitter, Sentiment analysis, Opinion mining, N...",sucesso,sucesso,não estruturado,6,6,6,6,6,2,1,2,2,2


In [9]:
# só mantem a galera com resultados selecionados no sample
df_slr1_google["flag_selecao"] = df_slr1_google["ID"].isin(df_slr1_resumo_google["ID"].to_list())
df_slr1_google = df_slr1_google[df_slr1_google["flag_selecao"] == True].copy()
df_slr1_google = df_slr1_google.drop(columns=["flag_selecao"])
df_slr1_google.shape

(47, 17)

### Antropic

In [10]:
df_slr1_titulo_resumo_antropic = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_claude_haiku_35_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_keywords_antropic = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_claude_haiku_35_slr1_v2.csv", encoding='utf-8')

df_slr1_titulo_keywords_antropic = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_claude_haiku_35_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_antropic = pd.read_csv("/data/codigos/dados/resultados/resumo_claude_haiku_35_slr1_v2.csv", encoding='utf-8')


In [11]:
# traz todos os resultados ja analisados na fase 1
df_slr1_antropic = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_claude_haiku_35_slr1.csv", encoding='utf-8')
df_slr1_antropic.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,claude-3-5-haiku-20241022_IC1_0,claude-3-5-haiku-20241022_IC1_1,claude-3-5-haiku-20241022_IC1_2,claude-3-5-haiku-20241022_IC1_3,claude-3-5-haiku-20241022_IC1_4,claude-3-5-haiku-20241022_IC2_0,claude-3-5-haiku-20241022_IC2_1,claude-3-5-haiku-20241022_IC2_2,claude-3-5-haiku-20241022_IC2_3,claude-3-5-haiku-20241022_IC2_4
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não estruturado,7,7,7,7,7,6,6,6,6,6
1,slr1_2,Sentiment analysis on arabic tweets: Challeng...,Sentiment analysis and opinion mining are desi...,"Twitter, Sentiment analysis, Opinion mining, N...",sucesso,sucesso,não estruturado,7,7,7,7,7,4,4,4,4,4


In [12]:
# só mantem a galera com resultados selecionados no sample
df_slr1_antropic["flag_selecao"] = df_slr1_antropic["ID"].isin(df_slr1_resumo_antropic["ID"].to_list())
df_slr1_antropic = df_slr1_antropic[df_slr1_antropic["flag_selecao"] == True].copy()
df_slr1_antropic = df_slr1_antropic.drop(columns=["flag_selecao"])
df_slr1_antropic.shape

(47, 17)

### META

In [13]:
df_slr1_titulo_resumo_meta = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_llama_4_scout_17B_16E_instruct_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_keywords_meta = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_llama_4_scout_17B_16E_instruct_slr1_v2.csv", encoding='utf-8')

df_slr1_titulo_keywords_meta = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_llama_4_scout_17B_16E_instruct_slr1_v2.csv", encoding='utf-8')

df_slr1_resumo_meta = pd.read_csv("/data/codigos/dados/resultados/resumo_llama_4_scout_17B_16E_instruct_slr1_v2.csv", encoding='utf-8')

In [14]:
# traz todos os resultados ja analisados na fase 1
df_slr1_meta = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_Llama-4-Scout-17B-16E-Instruct_slr1_v2.csv", encoding='utf-8')
df_slr1_meta.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,segunda_coleta,obs,Llama-4-Scout-17B-16E-Instruct_IC1_0,Llama-4-Scout-17B-16E-Instruct_IC1_1,Llama-4-Scout-17B-16E-Instruct_IC1_2,Llama-4-Scout-17B-16E-Instruct_IC1_3,Llama-4-Scout-17B-16E-Instruct_IC1_4,Llama-4-Scout-17B-16E-Instruct_IC2_0,Llama-4-Scout-17B-16E-Instruct_IC2_1,Llama-4-Scout-17B-16E-Instruct_IC2_2,Llama-4-Scout-17B-16E-Instruct_IC2_3,Llama-4-Scout-17B-16E-Instruct_IC2_4
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não,não estruturado,7,7,7,7,7,6,6,6,6,6
1,slr1_2,Sentiment analysis on arabic tweets: Challeng...,Sentiment analysis and opinion mining are desi...,"Twitter, Sentiment analysis, Opinion mining, N...",sucesso,sucesso,não,não estruturado,7,7,7,7,7,2,2,2,2,2


In [15]:
# só mantem a galera com resultados selecionados no sample
df_slr1_meta["flag_selecao"] = df_slr1_meta["ID"].isin(df_slr1_resumo_meta["ID"].to_list())
df_slr1_meta = df_slr1_meta[df_slr1_meta["flag_selecao"] == True].copy()
df_slr1_meta = df_slr1_meta.drop(columns=["flag_selecao"])
df_slr1_meta.shape

(50, 18)

### joins

In [16]:
# vamos trazer os dados disponibilizados pelos autores para comparação de performance
df_slr1_autores = pd.read_excel("/data/codigos/dados/SLR1-results-keys.xlsx",engine='openpyxl')
df_slr1_autores.head()

,ID,Title,IC1,IC2,Benchmark,Execution time (s),Total Tokens
0,slr1_1,User Experience_x000D_\n Design Using Machine...,7.0,6.0,I,1.321527,955.0
1,slr1_2,Sentiment analysis_x000D_\n on arabic tweets:...,7.0,4.0,E,0.922849,707.0
2,slr1_3,Machine learning_x000D_\n based cognitive ski...,7.0,6.0,E,1.285468,705.0
3,slr1_4,Human Factors_x000D_\n Engineering in Interac...,7.0,4.0,E,1.144713,783.0
4,slr1_5,Empirical comparisons_x000D_\n for combining ...,1.0,4.0,E,1.152106,1327.0


In [17]:
df_slr1_titulo_resumo_gpt = pd.merge(left=df_slr1_titulo_resumo_gpt, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_resumo_gpt.shape


(50, 16)

In [18]:
df_slr1_titulo_keywords_gpt = pd.merge(left=df_slr1_titulo_keywords_gpt, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_keywords_gpt.shape

(50, 16)

In [19]:
df_slr1_resumo_keywords_gpt = pd.merge(left=df_slr1_resumo_keywords_gpt, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_keywords_gpt.shape

(50, 16)

In [20]:
df_slr1_resumo_gpt = pd.merge(left=df_slr1_resumo_gpt, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_gpt.shape

(50, 15)

In [21]:
df_slr1_gpt = pd.merge(left=df_slr1_gpt, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_gpt.shape

(47, 20)

In [22]:
df_slr1_resumo_keywords_google = pd.merge(left=df_slr1_resumo_keywords_google, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_keywords_google.shape

(50, 16)

In [23]:
df_slr1_titulo_resumo_google = pd.merge(left=df_slr1_titulo_resumo_google, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_resumo_google.shape

(50, 16)

In [24]:
df_slr1_titulo_keywords_google = pd.merge(left=df_slr1_titulo_keywords_google, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_keywords_google.shape

(50, 16)

In [25]:
df_slr1_resumo_google = pd.merge(left=df_slr1_resumo_google, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_google.shape

(50, 25)

In [26]:
df_slr1_google = pd.merge(left=df_slr1_google, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_google.shape

(47, 20)

In [27]:
df_slr1_titulo_resumo_antropic = pd.merge(left=df_slr1_titulo_resumo_antropic, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_resumo_antropic.shape

(50, 16)

In [28]:
df_slr1_resumo_keywords_antropic = pd.merge(left=df_slr1_resumo_keywords_antropic, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_keywords_antropic.shape

(50, 16)

In [29]:
df_slr1_titulo_keywords_antropic = pd.merge(left=df_slr1_titulo_keywords_antropic, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_keywords_antropic.shape

(50, 16)

In [30]:
df_slr1_resumo_antropic = pd.merge(left=df_slr1_resumo_antropic, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_antropic.shape

(50, 25)

In [31]:
df_slr1_antropic = pd.merge(left=df_slr1_antropic, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_antropic.shape

(47, 20)

In [32]:
df_slr1_titulo_resumo_meta = pd.merge(left=df_slr1_titulo_resumo_meta, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_resumo_meta.shape

(50, 16)

In [33]:
df_slr1_resumo_keywords_meta = pd.merge(left=df_slr1_resumo_keywords_meta, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_keywords_meta.shape

(50, 16)

In [34]:
df_slr1_titulo_keywords_meta = pd.merge(left=df_slr1_titulo_keywords_meta, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_titulo_keywords_meta.shape

(50, 16)

In [35]:
df_slr1_resumo_meta = pd.merge(left=df_slr1_resumo_meta, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_resumo_meta.shape

(50, 25)

In [36]:
df_slr1_meta = pd.merge(left=df_slr1_meta, right=df_slr1_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr1_meta.shape

(50, 21)

### Geração dos resultados das llms e padronizacao dos campos

In [37]:
# criando a regra do corte em 5
def convert_benchmark(df,col1):
    df["result_bench"] = np.where(df[col1].str.lower()=="i",1,0)
    return df

In [38]:
dicto_resultados = {"df_slr1_titulo_resumo_gpt":df_slr1_titulo_resumo_gpt,
                    "df_slr1_titulo_keywords_gpt":df_slr1_titulo_keywords_gpt,
                    "df_slr1_resumo_keywords_gpt":df_slr1_resumo_keywords_gpt,
                    "df_slr1_resumo_gpt":df_slr1_resumo_gpt,
                    "df_slr1_gpt":df_slr1_gpt,
                    "df_slr1_resumo_keywords_google":df_slr1_resumo_keywords_google,
                    "df_slr1_titulo_resumo_google":df_slr1_titulo_resumo_google,
                    "df_slr1_titulo_keywords_google":df_slr1_titulo_keywords_google,
                    "df_slr1_resumo_google":df_slr1_resumo_google,
                    "df_slr1_google":df_slr1_google,
                    "df_slr1_titulo_resumo_antropic":df_slr1_titulo_resumo_antropic,
                    "df_slr1_resumo_keywords_antropic":df_slr1_resumo_keywords_antropic,
                    "df_slr1_titulo_keywords_antropic":df_slr1_titulo_keywords_antropic,
                    "df_slr1_resumo_antropic":df_slr1_resumo_antropic,
                    "df_slr1_antropic":df_slr1_antropic,
                    "df_slr1_titulo_resumo_meta":df_slr1_titulo_resumo_meta,
                    "df_slr1_resumo_keywords_meta":df_slr1_resumo_keywords_meta,
                    "df_slr1_titulo_keywords_meta":df_slr1_titulo_keywords_meta,
                    "df_slr1_resumo_meta":df_slr1_resumo_meta,
                    "df_slr1_meta":df_slr1_meta}

dicto_modelos = {"df_slr1_titulo_resumo_gpt":"gpt-4o",
                    "df_slr1_titulo_keywords_gpt":"gpt-4o",
                    "df_slr1_resumo_keywords_gpt":"gpt-4o",
                    "df_slr1_resumo_gpt":"gpt-4o",
                    "df_slr1_gpt":"gpt-4o",
                    "df_slr1_resumo_keywords_google":"gemini-1.5-flash",
                    "df_slr1_titulo_resumo_google":"gemini-1.5-flash",
                    "df_slr1_titulo_keywords_google":"gemini-1.5-flash",
                    "df_slr1_resumo_google":"gemini-1.5-flash",
                    "df_slr1_google":"gemini-1.5-flash",
                    "df_slr1_titulo_resumo_antropic":"claude-3-5-haiku-20241022",
                    "df_slr1_resumo_keywords_antropic":"claude-3-5-haiku-20241022",
                    "df_slr1_titulo_keywords_antropic":"claude-3-5-haiku-20241022",
                    "df_slr1_resumo_antropic":"claude-3-5-haiku-20241022",
                    "df_slr1_antropic":"claude-3-5-haiku-20241022",
                    "df_slr1_titulo_resumo_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr1_resumo_keywords_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr1_titulo_keywords_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr1_resumo_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr1_meta":"Llama-4-Scout-17B-16E-Instruct"}

dicto_selecao = {"df_slr1_titulo_resumo_gpt":"title+abstract",
                    "df_slr1_titulo_keywords_gpt":"title+keywords",
                    "df_slr1_resumo_keywords_gpt":"abstract+keywords",
                    "df_slr1_resumo_gpt":"abstract",
                    "df_slr1_gpt":"title+abstract+keywords",
                    "df_slr1_resumo_keywords_google":"abstract+keywords",
                    "df_slr1_titulo_resumo_google":"title+abstract",
                    "df_slr1_titulo_keywords_google":"title+keywords",
                    "df_slr1_resumo_google":"abstract",
                    "df_slr1_google":"title+abstract+keywords",
                    "df_slr1_titulo_resumo_antropic":"title+abstract",
                    "df_slr1_resumo_keywords_antropic":"abstract+keywords",
                    "df_slr1_titulo_keywords_antropic":"title+keywords",
                    "df_slr1_resumo_antropic":"abstract",
                    "df_slr1_antropic":"title+abstract+keywords",
                    "df_slr1_titulo_resumo_meta":"title+abstract",
                    "df_slr1_resumo_keywords_meta":"abstract+keywords",
                    "df_slr1_titulo_keywords_meta":"title+keywords",
                    "df_slr1_resumo_meta":"abstract",
                    "df_slr1_meta":"title+abstract+keywords"}


In [39]:
for key in dicto_resultados:
    dicto_resultados[key] = convert_benchmark(dicto_resultados[key], "Benchmark")

In [40]:
df_slr1_antropic.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,claude-3-5-haiku-20241022_IC1_0,claude-3-5-haiku-20241022_IC1_1,claude-3-5-haiku-20241022_IC1_2,claude-3-5-haiku-20241022_IC1_3,claude-3-5-haiku-20241022_IC1_4,claude-3-5-haiku-20241022_IC2_0,claude-3-5-haiku-20241022_IC2_1,claude-3-5-haiku-20241022_IC2_2,claude-3-5-haiku-20241022_IC2_3,claude-3-5-haiku-20241022_IC2_4,IC1,IC2,Benchmark,result_bench
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não estruturado,7,7,7,7,7,6,6,6,6,6,7.0,6.0,I,1
1,slr1_5,Empirical comparisons for combining balancing...,The process of modelling individual player per...,"Class imbalance, data mining, data sampling, f...",sucesso,sucesso,não estruturado,1,1,1,1,1,2,2,2,2,2,1.0,4.0,E,0


In [41]:
def create_result_llm_classic(df,col1,col2,limit1,limit2,col_name):
    """
    Create flag with result for llms
    """
    df["aux_ic1"] = np.where(df[col1]>=limit1,1,0)
    df["aux_ic2"] = np.where(df[col2]>=limit2,1,0)
    df[col_name] = np.where(((df["aux_ic1"]==1) & (df["aux_ic2"]==1)),1,0)
    return df.drop(columns=["aux_ic1","aux_ic2"])

In [42]:
# Cria a resposta final para cada iteração
def calculate_select_result_llm(n_interactions, model_gpt, df):
    name_ic1 = "IC1"
    name_ic2 = "IC2"
    for i in range(n_interactions):
        coluna_aux1 = model_gpt+"_"+name_ic1+"_"+str(i)
        coluna_aux2 = model_gpt+"_"+name_ic2+"_"+str(i)
        col_name = "result_llm"+"_"+str(i)
        df = create_result_llm_classic(df=df,
                                        col1=coluna_aux1,
                                        col2=coluna_aux2,
                                        limit1=5,
                                        limit2=5,
                                        col_name=col_name)
    return df

In [43]:
n_interactions = 5

In [44]:
for key in dicto_resultados:
    print(key)
    dicto_resultados[key] = calculate_select_result_llm(n_interactions=n_interactions, 
                                                        model_gpt=dicto_modelos[key],
                                                        df=dicto_resultados[key])

df_slr1_titulo_resumo_gpt
df_slr1_titulo_keywords_gpt
df_slr1_resumo_keywords_gpt
df_slr1_resumo_gpt
df_slr1_gpt
df_slr1_resumo_keywords_google
df_slr1_titulo_resumo_google
df_slr1_titulo_keywords_google
df_slr1_resumo_google
df_slr1_google
df_slr1_titulo_resumo_antropic
df_slr1_resumo_keywords_antropic
df_slr1_titulo_keywords_antropic
df_slr1_resumo_antropic
df_slr1_antropic
df_slr1_titulo_resumo_meta
df_slr1_resumo_keywords_meta
df_slr1_titulo_keywords_meta
df_slr1_resumo_meta
df_slr1_meta


In [45]:
df_slr1_titulo_resumo_meta.head(2)


,ID,title,abstract,Llama-4-Scout-17B-16E-Instruct_IC1_0,Llama-4-Scout-17B-16E-Instruct_IC1_1,Llama-4-Scout-17B-16E-Instruct_IC1_2,Llama-4-Scout-17B-16E-Instruct_IC1_3,Llama-4-Scout-17B-16E-Instruct_IC1_4,Llama-4-Scout-17B-16E-Instruct_IC2_0,Llama-4-Scout-17B-16E-Instruct_IC2_1,Llama-4-Scout-17B-16E-Instruct_IC2_2,Llama-4-Scout-17B-16E-Instruct_IC2_3,Llama-4-Scout-17B-16E-Instruct_IC2_4,IC1,IC2,Benchmark,result_bench,aux_ic1,aux_ic2,result_llm_0
0,slr1_77,A Literature Review of AR-Based Remote Guidan...,The future of work is increasingly mobile and ...,6,6,6,6,6,5,5,5,5,5,7.0,6.0,I,1,1,1,1
1,slr1_21,User-centered design to improve clinical deci...,Background: A growing literature has demonstra...,1,1,1,1,1,2,2,2,2,2,1.0,4.0,E,0,0,0,0


In [46]:
dicto_resultados["df_slr1_titulo_resumo_meta"].head(2)

,ID,title,abstract,Llama-4-Scout-17B-16E-Instruct_IC1_0,Llama-4-Scout-17B-16E-Instruct_IC1_1,Llama-4-Scout-17B-16E-Instruct_IC1_2,Llama-4-Scout-17B-16E-Instruct_IC1_3,Llama-4-Scout-17B-16E-Instruct_IC1_4,Llama-4-Scout-17B-16E-Instruct_IC2_0,Llama-4-Scout-17B-16E-Instruct_IC2_1,Llama-4-Scout-17B-16E-Instruct_IC2_2,Llama-4-Scout-17B-16E-Instruct_IC2_3,Llama-4-Scout-17B-16E-Instruct_IC2_4,IC1,IC2,Benchmark,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4
0,slr1_77,A Literature Review of AR-Based Remote Guidan...,The future of work is increasingly mobile and ...,6,6,6,6,6,5,5,5,5,5,7.0,6.0,I,1,1,1,1,1,1
1,slr1_21,User-centered design to improve clinical deci...,Background: A growing literature has demonstra...,1,1,1,1,1,2,2,2,2,2,1.0,4.0,E,0,0,0,0,0,0


### Ajustando os dataframes

In [47]:

for key in dicto_resultados:
    print(key)
    old=dicto_modelos[key]
    new="llm"
    dicto_resultados[key]["model_id"]  = dicto_modelos[key]
    dicto_resultados[key]["set_features"]  = dicto_selecao[key]
    dicto_resultados[key]["slr"]  = "slr1"
    dicto_resultados[key].columns = dicto_resultados[key].columns.str.replace(old, new, regex=True)

df_slr1_titulo_resumo_gpt
df_slr1_titulo_keywords_gpt
df_slr1_resumo_keywords_gpt
df_slr1_resumo_gpt
df_slr1_gpt
df_slr1_resumo_keywords_google
df_slr1_titulo_resumo_google
df_slr1_titulo_keywords_google
df_slr1_resumo_google
df_slr1_google
df_slr1_titulo_resumo_antropic
df_slr1_resumo_keywords_antropic
df_slr1_titulo_keywords_antropic
df_slr1_resumo_antropic
df_slr1_antropic
df_slr1_titulo_resumo_meta
df_slr1_resumo_keywords_meta
df_slr1_titulo_keywords_meta
df_slr1_resumo_meta
df_slr1_meta


In [48]:
dicto_resultados["df_slr1_meta"].head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,segunda_coleta,obs,llm_IC1_0,llm_IC1_1,llm_IC1_2,llm_IC1_3,llm_IC1_4,llm_IC2_0,llm_IC2_1,llm_IC2_2,llm_IC2_3,llm_IC2_4,IC1,IC2,Benchmark,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4,model_id,set_features,slr
0,slr1_1,User Experience Design Using Machine Learning:...,ser experience (UX) is the key to increased pr...,"User experience, experience design, UX, ED, ma...",sucesso,sucesso,não,não estruturado,7,7,7,7,7,6,6,6,6,6,7.0,6.0,I,1,1,1,1,1,1,Llama-4-Scout-17B-16E-Instruct,title+abstract+keywords,slr1
1,slr1_5,Empirical comparisons for combining balancing...,The process of modelling individual player per...,"Class imbalance, data mining, data sampling, f...",sucesso,sucesso,não,não estruturado,2,2,2,2,2,2,2,2,2,2,1.0,4.0,E,0,0,0,0,0,0,Llama-4-Scout-17B-16E-Instruct,title+abstract+keywords,slr1


In [49]:
lista_colunas_indentificadores = ["ID","slr","model_id","set_features"]
lista_colunas_ics = ["IC1","IC2","llm_IC1_0","llm_IC2_0","llm_IC1_1","llm_IC2_1","llm_IC1_2",
                     "llm_IC2_2","llm_IC1_3","llm_IC2_3","llm_IC1_4","llm_IC2_4"]
lista_colunas_resultados = ["result_bench","result_llm_0","result_llm_1","result_llm_2","result_llm_3","result_llm_4"]

In [50]:
# empilha os resultados em uma unico dataframe
df_resultados_empilhados = pd.DataFrame()
for key in dicto_resultados:
    df_temp = dicto_resultados[key][lista_colunas_indentificadores + lista_colunas_ics + lista_colunas_resultados].copy()
    df_resultados_empilhados = pd.concat([df_resultados_empilhados, df_temp], axis=0, ignore_index=True)

In [51]:
df_resultados_empilhados.head()

,ID,slr,model_id,set_features,IC1,IC2,llm_IC1_0,llm_IC2_0,llm_IC1_1,llm_IC2_1,llm_IC1_2,llm_IC2_2,llm_IC1_3,llm_IC2_3,llm_IC1_4,llm_IC2_4,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4
0,slr1_77,slr1,gpt-4o,title+abstract,7.0,6.0,6,5,6,5,6,5,6,5,6,5,1,1,1,1,1,1
1,slr1_21,slr1,gpt-4o,title+abstract,1.0,4.0,3,4,3,4,3,3,3,4,3,4,0,0,0,0,0,0
2,slr1_123,slr1,gpt-4o,title+abstract,7.0,6.0,6,3,6,3,5,3,6,3,5,3,0,0,0,0,0,0
3,slr1_71,slr1,gpt-4o,title+abstract,1.0,5.0,1,4,3,4,1,4,1,4,3,4,0,0,0,0,0,0
4,slr1_99,slr1,gpt-4o,title+abstract,7.0,7.0,6,6,6,6,6,6,6,6,6,6,1,1,1,1,1,1


In [52]:
df_resultados_empilhados.shape

(991, 22)

In [53]:
df_resultados_empilhados["model_id"].unique()

array(['gpt-4o', 'gemini-1.5-flash', 'claude-3-5-haiku-20241022',
       'Llama-4-Scout-17B-16E-Instruct'], dtype=object)

In [54]:
# vendo se os indentificadores juntos formam uma chave unica
df_resultados_empilhados["chave_unica"] = df_resultados_empilhados["ID"].astype(str) + "_" + df_resultados_empilhados["slr"].astype(str) + "_" + df_resultados_empilhados["model_id"].astype(str) + "_" + df_resultados_empilhados["set_features"].astype(str) 
df_resultados_empilhados["chave_unica"].nunique() == df_resultados_empilhados.shape[0]


True

In [55]:
# deleta chave unica
df_resultados_empilhados = df_resultados_empilhados.drop(columns=["chave_unica"])
df_resultados_empilhados.shape

(991, 22)

In [56]:
# grava os resultados em excel e csv
df_resultados_empilhados.to_excel("/data/codigos/dados/resultados/resultados_empilhados_llms_slr1_prep_GLMM_v2.xlsx", index=False)
df_resultados_empilhados.to_csv("/data/codigos/dados/resultados/resultados_empilhados_llms_slr1_prep_GLMM_v2.csv", index=False, encoding='utf-8')
